In [2]:
import os
import pandas as pd
import numpy as np
import gc

from datetime import datetime, timedelta
from astropy.time import Time

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

* To see the arquive lenght in CMD use: **wc -l arquivename.csv**
* To see the arquive columns in CMD use: **head -n 1 arquivename.csv**

## Functions

In [3]:
def filter_valid_glitches(unclustered, clustered, window_T, min_triggers, fmin=3, fmax=8192):
    times_all = unclustered['time'].values
    indices_validos = []

    for i, row in enumerate(clustered.itertuples(index=False)):
        t_center = row.time
    
        tmin = t_center - window_T / 2
        tmax = t_center + window_T / 2

        i0 = np.searchsorted(times_all, tmin, side="left")
        i1 = np.searchsorted(times_all, tmax, side="right")
        
        # Conta quantos triggers caem dentro da janela do glitch
        n_triggers = i1 - i0
        
        # Mantém apenas se tiver 40 ou mais triggers
        if n_triggers >= min_triggers:
            indices_validos.append(i)
    
    # Retorna apenas as linhas do clustered que passaram no filtro
    return clustered.iloc[indices_validos].reset_index(drop=True)

In [4]:
def filter_intervals(triggers, gps_start, gps_end, intervals):
    
    intervals_crop = intervals[(intervals['int2'] >= gps_start) & (intervals['int1'] <= gps_end)].copy().sort_values('int1')
    intervals_crop['int1'] = intervals_crop['int1'].astype('float64')

    triggers_crop = triggers[(triggers['time'] >= gps_start) & (triggers['time'] <= gps_end)].copy().sort_values('time')

    if triggers_crop.empty or intervals_crop.empty:
        return pd.DataFrame(columns=triggers.columns)

    merged = pd.merge_asof(triggers_crop, intervals_crop, left_on='time', right_on='int1', direction='backward')
    # valid = merged[(merged['time'] >= merged['int1']) & (merged['time'] <= merged['int2'])]
    valid = merged[merged['time'] <= merged['int2']].copy()

    triggers_final = valid.drop(columns=['int1', 'int2'], errors='ignore')

    return triggers_final

In [187]:
def glitch_to_glitchgram(unclustered, clustered, min_triggers, labels, window_T=0.2, fmin=3, fmax=8192, tbins=41, fbins=30):
    """
    This function create one glitchgram from unclustered Omicron data, for each glitch.

    Parameters:
        df_triggers (dataFrame): A dataFrame of GPStimes of all triggers inside of some window.
        tmin, tmax (numbers): limits of GPStimes that contain all triggers of some glitch.
        fmin, fmax (numbers): frequency range for the glitchgram.
        n_time_bins (int): number of time bins.
        n_freq_bins (int): number of frequency bins.
        norm (boolean): True if you need normalize SNR values.

    Returns:
        array for some glitch: each entry is a SNR value from flattened (binsf × binst) matrix.
    """

    times_all = unclustered['time'].values
    freqs_all = unclustered['frequency'].values
    snrs_all  = unclustered['snr'].values
    
    vecs = []
    labs = []
    GPSs = []

    time_edges = np.linspace(-window_T/2, window_T/2, tbins+1)
    freq_edges = np.logspace(np.log10(fmin), np.log10(fmax), fbins+1)
    
    for i, row in enumerate(clustered.itertuples(index=False)):

        t_center = row.time
    
        tmin = t_center - window_T/2
        tmax = t_center + window_T/2

        i0 = np.searchsorted(times_all, tmin, side="left")
        i1 = np.searchsorted(times_all, tmax, side="right")
        
        times = times_all[i0:i1] - t_center
        freqs = freqs_all[i0:i1]
        snrs  = snrs_all[i0:i1]
        
        '''if len(times) < min_triggers:
            continue'''
    
        df_triggers = pd.DataFrame({'time': times,'frequency': freqs,'snr': snrs})
        
        t_idx = np.digitize(df_triggers['time'].values, time_edges) - 1
        f_idx = np.digitize(df_triggers['frequency'].values, freq_edges) - 1
            
        # initial empty matrix
        A = np.zeros((fbins, tbins), dtype=float)
    
        for ti, fi, si in zip(t_idx, f_idx, snrs):
            if 0 <= ti < tbins and 0 <= fi < fbins:
                if si > A[fi, ti]:
                    A[fi, ti] = si
                    
        '''# normalização do SNR 
        if len(snrs) > 0:
            snr_min = snrs.min()
            snr_max = snrs.max()
            if snr_max > snr_min:
                norm_snrs = (snrs - snr_min) / (snr_max - snr_min)
            else:
                norm_snrs = np.zeros_like(snrs)
        else:
            norm_snrs = snrs'''
        
        vecs.append(A.flatten(order="C"))
        GPSs.append(row.time)
        if labels: labs.append(row.label)

    vecs_final = np.array(vecs).astype('float32')
    GPSs_final = np.array(GPSs).astype('float32')
    if labels: labs_final = np.array(labs)

    if labels:
        return vecs_final, labs_final, GPSs_final
    else:
        return vecs_final, GPSs_final

## Parameters

In [180]:
window_T = 2.0       # window around the glitch
deltat = 0.2         # time btw triggers from the same cluster 

cutSNR = True
SNRvalue_cut = 5.0
min_triggers = 20
gap_time = 1.0        # time btw possible duplicated glitches
threshold_snr = 0.5

filterr = True

In [181]:
run = 'O3a'
clust = 'teste5_duplicate_glitches'
arqType = "SNRclustered"

/////////////////////////////////////////////////////////////////////////////////////

In [176]:
base = "/home/esanches/Projetos/tSNE/Virgo"
repository = "/home/esanches/Projetos/tSNE/Virgo/zrepository"
save_file = "/home/esanches/Projetos/tSNE/Virgo/zrepository/" + clust + "/" + run

suffix = "_C" + str(min_triggers) + "_dT" + str(deltat) + "_SNR" + str(SNRvalue_cut) + "_gt" + str(gap_time) + "_th" + str(threshold_snr)

In [177]:
gspy_arquive = save_file + "/" + "valid_clustered" + "_C" + str(min_triggers) + ".csv"
gdchar_arquive = save_file + "/" + arqType + suffix + ".csv"

In [178]:
gdchar_arquive

'/home/esanches/Projetos/tSNE/Virgo/zrepository/teste5_duplicate_glitches/O3a/SNRclustered_C20_dT0.2_SNR5.0_gt1.0_th0.5.csv'

In [100]:
teste_path = repository + "/gspy/" + run + "/valid_clustered_" + run + "_C" + str(min_triggers) + ".csv"

In [101]:
teste_path

'/home/esanches/Projetos/tSNE/Virgo/zrepository/gspy/O3a/valid_clustered_O3a_C20.csv'

In [14]:


arq_name = 'SNRclustered' + suffix if cutSNR else 'clustered' + suffix
clustered_path = "/home/esanches/Projetos/tSNE/Virgo/zrepository/" + clust + "/" + run + "/" + arq_name + ".csv"




vecs_path = "/home/esanches/Projetos/tSNE/Virgo/zrepository" + "/" + clust + "/" + run

In [55]:
# suffix = "_filtered" if filterr else "_"

# clust = 'teste4_intervals'
# suffix = "_C" + str(min_triggers) + "_dT" + str(deltat) + "_SNR" + str(SNRvalue_cut)

# clust = 'teste5_duplicate_glitches'
# suffix = "_C" + str(min_triggers) + "_dT" + str(deltat) + "_dF" + str(window_F) + "_SNR" + str(SNRvalue_cut) # + "_gt" + str(gap_time) + "_th" + str(threshold_snr)

# clust = 'teste7_2Dclustering'

In [56]:
repository = "/home/esanches/Projetos/tSNE/Virgo/zrepository"


vectors_path =  teste_path + "/" + "tSNEandHDBSCAN"

daily_clustering = "/home/esanches/Projetos/tSNE/Virgo/zrepository/" + clust + "/" + run + "/" + "dailyClustering" + suffix
clustered_path = "/home/esanches/Projetos/tSNE/Virgo/zrepository/" + clust + "/" + run + "/" + arq_name + ".csv"
vecs_save = "/home/esanches/Projetos/tSNE/Virgo/zrepository/" + clust + "/" + run + "/" + "vecs" + suffix + ".npy"

# non_matched_gdchar_name = vectors_path + "/non_matched_gdchar" + suffix + ".csv"

## Triggers

In [10]:
trigcols = ['time', 'frequency', 'snr']

dtypes = {
    'time': 'float64',
    'frequency': 'float32',
    'snr': 'float32'
}

source_triggers = base + "/" + "triggers_total_" + run + ".csv"

chunk_size = 10_000_000
count = 0
chunks = []

for chunk in pd.read_csv(source_triggers, usecols=trigcols, dtype=dtypes, chunksize=chunk_size):
    print(f'Reading chunk: {count}')
    chunks.append(chunk)
    count += 1

    del chunk
    gc.collect()

triggers = pd.concat(chunks, ignore_index=True)
triggers = triggers.sort_values('time').reset_index(drop=True)

unclustered_all = triggers.copy()

Reading chunk: 0
Reading chunk: 1
Reading chunk: 2
Reading chunk: 3
Reading chunk: 4
Reading chunk: 5
Reading chunk: 6
Reading chunk: 7
Reading chunk: 8
Reading chunk: 9
Reading chunk: 10
Reading chunk: 11
Reading chunk: 12
Reading chunk: 13
Reading chunk: 14
Reading chunk: 15


## Gravity Spy data

### Create gspy clustering

In [90]:
runs_dict = {
    'O3a': {
        'start': datetime(2019, 4, 1, 0, 0, 0),
        'end':   datetime(2019, 10, 1, 15, 0, 0)
    },
    'O3b': {
        'start': datetime(2019, 11, 1, 15, 0, 0),
        'end':   datetime(2020, 3, 27, 17, 0, 0)
    }
}

data_start = runs_dict[run]['start']
data_end   = runs_dict[run]['end']

gps_start = Time(data_start, scale="utc").gps
gps_end   = Time(data_end, scale="utc").gps

intervals = pd.read_csv(repository + '/gwosc' + run + '.txt', sep=' ', header=None, names=['int1', 'int2', 'duration'])

In [91]:
chunk_size = 10000000

source_triggers = base + "/" + "triggers_total_" + run + ".csv"
source_glitches = base + "/gspy" + run + ".csv"

clustered = pd.read_csv(source_glitches)
clustered = clustered.rename(columns={'GPStime':'time'})

count = 0
chunks = []

for chunk in pd.read_csv(source_triggers, chunksize=chunk_size):
    print(f'Processing chunk: {count}')
    count += 1

    filtered_chunk = filter_intervals(chunk, gps_start, gps_end, intervals)

    if not filtered_chunk.empty:

        valid_clustered_chunk = filter_valid_glitches(filtered_chunk, clustered, window_T, min_triggers)
        chunks.append(valid_clustered_chunk)
    
        del chunk, filtered_chunk
        gc.collect()

valid_clustered = pd.concat(chunks, ignore_index=True)

valid_clustered = valid_clustered[valid_clustered['confidence'] >= 0.9]
valid_clustered = valid_clustered.drop(valid_clustered[valid_clustered['label'].isin(['None_of_the_Above', 'No_Glitch'])].index).reset_index(drop=True)

valid_clustered.to_csv(gspy_arquive, index=False)

Processing chunk: 0
Processing chunk: 1
Processing chunk: 2
Processing chunk: 3
Processing chunk: 4
Processing chunk: 5
Processing chunk: 6
Processing chunk: 7
Processing chunk: 8
Processing chunk: 9
Processing chunk: 10
Processing chunk: 11
Processing chunk: 12
Processing chunk: 13
Processing chunk: 14
Processing chunk: 15


### Create gspy vectors

In [83]:
gspy = pd.read_csv(gspy_arquive)

# gspy = gspy.rename(columns={'GPStime':'time'})
gspy = gspy.sort_values(by=['time']).reset_index(drop=True)

if filterr == True:
    top10 = gspy['label'].value_counts().head(10).index
    gspy_top10 = gspy[gspy['label'].isin(top10)].copy()
    
    limited_glit = gspy_top10.groupby('label', group_keys=False).apply(lambda x: x.sample(n=min(len(x), 1000), random_state=42)).reset_index(drop=True)
    limited_glit = limited_glit.sort_values(by=['time']).reset_index(drop=True)
    
    clustered = limited_glit.copy()
else:
    clustered = gspy.copy()

clustered.tail()

,time,peakFreq,snr,amplitude,centralFreq,duration,bandwidth,chisq,chisqDof,confidence,id,ifo,label,imgUrl,Q-value
8739,1.253962e+09,22.647,7.527,8.400000e-22,2837.936,1.375,5639.855469,0.0,0.0,0.921,mnyBwwcdWT,V1,Scattered_Light,https://ldas-jobs.ligo.caltech.edu/~gravityspy...,45.255
8740,1.253964e+09,49.200,7.503,1.510000e-22,50.045,0.297,39.862202,0.0,0.0,0.966,jzxl5Szk4n,V1,Repeating_Blips,https://ldas-jobs.ligo.caltech.edu/~gravityspy...,5.657
8741,1.253971e+09,18.731,7.899,1.710000e-21,20.195,0.875,8.390090,0.0,0.0,0.998,mbOHAEQ2zi,V1,Low_Frequency_Lines,https://ldas-jobs.ligo.caltech.edu/~gravityspy...,22.627
8742,1.253974e+09,24.104,9.501,9.300000e-22,26.590,1.000,21.179770,0.0,0.0,0.996,B9sr97s6ss,V1,Low_Frequency_Burst,https://ldas-jobs.ligo.caltech.edu/~gravityspy...,22.627
8743,1.253977e+09,20.168,9.750,1.910000e-21,3983.241,4.748,7934.482910,0.0,0.0,0.951,elTP3iCl5U,V1,Scattered_Light,https://ldas-jobs.ligo.caltech.edu/~gravityspy...,22.627


In [84]:
dataFrame = clustered.copy()

vecs, labs, GPSs = glitch_to_glitchgram(unclustered_all, dataFrame, min_triggers, labels)
np.save(save_file + "/vecs_C" + str(min_triggers), vecs)
np.save(save_file + "/labs_C" + str(min_triggers), labs)
np.save(save_file + "/GPSs_C" + str(min_triggers), GPSs)

## Detchar clustering

In [188]:
gdchar_arquive

'/home/esanches/Projetos/tSNE/Virgo/zrepository/teste5_duplicate_glitches/O3a/SNRclustered_C20_dT0.2_SNR5.0_gt1.0_th0.5.csv'

In [189]:
labels = False

gdchar = pd.read_csv(gdchar_arquive)

print(len(gdchar))

if labels == True:
    vecs, labs, GPSs = glitch_to_glitchgram(unclustered_all, gdchar, min_triggers, labels)
else:
    vecs, GPSs = glitch_to_glitchgram(unclustered_all, gdchar, min_triggers, labels)
    
sps = len(vecs)

print(sps)

np.random.seed(42)
indices_aleatorios = np.random.choice(len(vecs), size=sps, replace=False)

vecs = vecs[indices_aleatorios]
GPSs = GPSs[indices_aleatorios]
if labels == True: labs = labs[indices_aleatorios]

np.save(save_file + "/vecs" + "_" + arqType + str(sps), vecs)
np.save(save_file + "/GPSs" + "_" + arqType + str(sps), GPSs)
if labels == True: np.save(save_file + "/labs" + "_" + arqType + str(sps), labs)

233716
233716


In [58]:
# data = data[data['confidence'] >= 0.9]
# data = data.drop(data[data['label'].isin(['None_of_the_Above', 'No_Glitch'])].index).reset_index(drop=True)

'''top10 = data['label'].value_counts().head(10).index
data_top10 = data[data['label'].isin(top10)].copy()

limited_glit = data_top10.groupby('label', group_keys=False).apply(lambda x: x.sample(n=min(len(x), 1000), random_state=42)).reset_index(drop=True)
limited_glit = limited_glit.sort_values(by=['time']).reset_index(drop=True)
        
clustered = limited_glit.copy()

vecs, labs, GPSs = glitch_to_glitchgram(unclustered_all_O3a, clustered, min_triggers, labels)

np.save(save_file + "/vecs" + suffix + "_" + arqType, vecs)
np.save(save_file + "/labs" + suffix + "_" + arqType, labs)
np.save(save_file + "/GPSs" + suffix + "_" + arqType, GPSs)'''



## vectorization

In [ ]:
labels



In [ ]:
if labels:

else:
    vecs, GPSs = glitch_to_glitchgram(unclustered_all_O3a, dataFrame, min_triggers, labels)
    np.save(save_file + "_vecs", vecs)
    np.save(save_file + "_GPSs", GPSs)

In [43]:
labels = True

dataFrame = pd.read_csv(arquive)
dataFrame

,time,peakFreq,snr,amplitude,centralFreq,duration,bandwidth,chisq,chisqDof,confidence,id,ifo,label,imgUrl,Q-value
70222,1.241234e+09,41.312,12.682,3.250000e-22,63.212,0.250,86.914932,0.0,0.0,0.934,JhpNVMS0cT,V1,Tomte,https://ldas-jobs.ligo.caltech.edu/~gravityspy...,5.657
70223,1.241234e+09,95.999,17.450,1.770000e-22,841.317,0.161,1622.407349,0.0,0.0,1.000,bbmK3H6Wp0,V1,Blip,https://ldas-jobs.ligo.caltech.edu/~gravityspy...,5.657
70224,1.241236e+09,62.976,11.238,1.530000e-22,136.131,0.156,223.481537,0.0,0.0,0.967,t0altDCAq3,V1,Blip,https://ldas-jobs.ligo.caltech.edu/~gravityspy...,5.657
70225,1.241240e+09,95.999,12.868,1.270000e-22,1785.930,0.309,3498.642578,0.0,0.0,1.000,rjuJozERP2,V1,Blip,https://ldas-jobs.ligo.caltech.edu/~gravityspy...,5.657
70226,1.241241e+09,41.312,10.187,2.670000e-22,63.212,0.125,86.914932,0.0,0.0,0.737,F46nBi2XDS,V1,Tomte,https://ldas-jobs.ligo.caltech.edu/~gravityspy...,5.657


In [ ]:
save_file + "_GPSs", GPSs

NameError: name 'unclustered_all_O3a' is not defined

In [ ]:
matched.save(save_file + "/", "matched" + suffix)
non_matched_gspy.save(save_file + "/", "non_matched_gspy" + suffix)
non_matched_gdchar.save(save_file + "/", "non_matched_gdchar" + suffix)